In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from preprocessing import read_flow
fcs_dir = str(input("Path"))
tissue_type = input(str("Tissue type"))
df_flow, sample_list, session = read_flow(fcs_dir)

In [ ]:
exclude = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H',
           'SSC-H', 'AF-A', 'CD66bCD19CD326LD', 'Time', 'CD45']
df_flow = df_flow.drop(columns=exclude)

In [ ]:
df_flow_counts = df_flow.select_dtypes(include=[np.number])

In [ ]:
from preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
sc.pp.scale(adata, max_value=5)

In [ ]:
from preprocessing import population_filter
CD3 = population_filter(adata, 'CD3', 0)

In [ ]:
adata = adata[(adata[:, 'CD3'].X > 0)]
adata = adata[:, adata.var.index != 'CD3']

In [ ]:
CD4 = population_filter(adata, 'CD4', 0)

In [ ]:
CD8 = population_filter(adata, 'CD8', 0)

In [ ]:
gdTCR = population_filter(adata, 'gdTCR', 0)

In [ ]:
TCRva = population_filter(adata, 'TCRva', 0)

In [ ]:
from run_pipeline import clustering_pipeline
CD3 = clustering_pipeline(CD3, tissue_type, 'CD3')

In [ ]:
from run_pipeline import dem_ranked
CD3, unique_values = dem_ranked(CD3)

In [ ]:
# celltype = {'celltype': []}
# cluster_to_genes = {
#     '0': 'CD45RA+CD4+CCR7+ (0)',
#     '1': 'CD45RA+CD8+ (1)',
#     '2': 'CD45RA+CCR7+ (2)',
#     '3': 'CD45RA+CCR7+IL33R+ (3)',
#     '4': 'CD69+CD4+ (4)',
#     '5': 'CD4+ (5)',
#     '6': 'IL33R+ (6)',
#     '7': 'FOXP3+CD25+CD4+CD28+IL33R+ (7)',
#     '8': 'CXCR3+CD4+ (8)',
#     '9': 'CD69+CD4+PD1+ (9)',
#     '10': 'CXCR5+CD4+ (10)',
#     '11': 'CD4+CD28+IL33R+ (11)',
#     '12': 'CD45RA+CXCR6+IL33R+ (12)',
#     '13': 'CD69+CD4+PD1+CXCR6+ (13)',
#     '14': 'CD69+PD1+CD57+ (14)',
#     '15': 'CD103+CXCR3+CD69+CD8+ (15)',
#     '16': 'RORgt+ (16)',
#     '17': 'CD4+CRTH2+ (17)'

# }
# celltype['celltype'] = [cluster_to_genes[leiden]
#                         for leiden in sample.obs['leiden']]
# sample.obs["celltype"] = celltype['celltype']

# print(sample.obs[["leiden", "celltype"]].head())

In [ ]:
from plotting_methods import annotated_umap

annotated_umap(CD3, "All Tissues", "CD3", obs='celltype')

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = CD3[CD3.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['celltype'],
        title=f'{tissue_type} {"CD3"} {group} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12})